# Lab 2F: Evaluation in Python

**Time**: ~15 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will evaluate a RAG pipeline using the LLM-as-judge pattern.

## Prerequisites

- Python 3.10+ with `azure-cosmos`, `azure-identity`, `openai`, `numpy`, and `python-dotenv` installed: `pip install azure-cosmos azure-identity openai numpy python-dotenv`
- `COSMOS_ENDPOINT` environment variable set to your Cosmos DB account endpoint
- `FOUNDRY_ENDPOINT` environment variable set to your Foundry endpoint for chat completions
- `COMPLETIONS_MODEL` environment variable set to the name of the Foundry model for chat completions
- **Lab 2E must have already been run** in this Cosmos account. 2F's mock retrieval reads from the `WorkshopData.Docs` container with `partitionKey = 'rag'`, which is seeded by 2E. Without that data the judge has no context to score against.

Run each cell in order to complete the steps.

## Step 0: Initialize Connection

Set up the Cosmos client connection and Azure OpenAI clients.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
FOUNDRY_ENDPOINT = os.environ.get("FOUNDRY_ENDPOINT")
DB_NAME = "WorkshopData"
CONT_NAME = "Docs"
COMPLETIONS_MODEL = os.environ.get("COMPLETIONS_MODEL", "gpt41")
EVAL_MODEL = os.environ.get("EVAL_MODEL", COMPLETIONS_MODEL)

# This lab focuses on LLM-as-judge scoring; retrieval is mocked with a plain
# Cosmos query (see generate_response), so no embeddings client is needed —
# real embedding + vector search is covered in Labs 2D and 2E.
for var in ["COSMOS_ENDPOINT", "FOUNDRY_ENDPOINT"]:
    if not os.environ.get(var):
        raise RuntimeError(f"{var} environment variable is required.")

print(f"Cosmos Endpoint:   {ENDPOINT}")
print(f"Foundry Endpoint:  {FOUNDRY_ENDPOINT}")
print(f"Database:          {DB_NAME}")
print(f"Completions Model: {COMPLETIONS_MODEL}")
print(f"Eval Model:        {EVAL_MODEL}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import AzureCliCredential, get_bearer_token_provider
from openai import OpenAI

cred = AzureCliCredential()
cosmos_client = CosmosClient(url=ENDPOINT, credential=cred)
db = cosmos_client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}/{CONT_NAME}")

# Chat completions: Foundry endpoint, Entra ID auth.
token_provider = get_bearer_token_provider(cred, "https://ai.azure.com/.default")
foundry_client = OpenAI(
    base_url=f"{FOUNDRY_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)
print("Foundry chat client initialized")

## Step 1: Create Evaluation Dataset (Prebuilt)

Prints the three Q/A test cases used to evaluate the RAG pipeline.

In [ ]:
eval_dataset = [
    {"question": "What is Azure Cosmos DB?", "ground_truth": "Azure Cosmos DB is a globally distributed, multi-model database service from Microsoft."},
    {"question": "What types of indexes does Cosmos DB support?", "ground_truth": "Range, spatial, composite, vector, and full-text indexes."},
    {"question": "How do vector indexes differ from range indexes?", "ground_truth": "Vector indexes enable semantic similarity search on embeddings, while range indexes optimize numeric/string equality and ordering queries."}
]

print(f"Created {len(eval_dataset)} evaluation examples:")
for example in eval_dataset:
    print(f"  Q: {example['question']}")
    print(f"  Ground truth: {example['ground_truth']}\n")

## Step 2: Score RAG Outputs

The retrieval call, the chat-judge call, the score parser, and the result aggregation are all already wired up. Your job is to write the **judge prompt** that turns the chat model into an evaluator.

Replace the placeholder `scoring_prompt` in the code cell with a real LLM-as-judge prompt:

```python
scoring_prompt = (
    "Rate the answer's relevance to the question on a scale of 1 to 5.\n"
    "Respond with ONLY a single digit (1, 2, 3, 4, or 5). No words, no punctuation, no explanation.\n\n"
    f"Question: {example['question']}\n"
    f"Answer: {answer}\n"
    f"Ground truth: {example['ground_truth']}\n\n"
    "Score (single digit only):"
)
```

**Expected output**: A 1–5 relevance score per question, then an average. The placeholder prompt asks for `0` so the 1–5 parser falls through — that is the signal that the placeholder has not been replaced.

In [ ]:
import re


def generate_response(question: str) -> str:
    result = container.query_items(
        query="SELECT TOP 3 c.text FROM c WHERE c.partitionKey = 'rag'",
        partition_key="rag"
    )
    context = "\n\n".join(r.get("text", "") for r in result)
    completion = foundry_client.chat.completions.create(
        model=COMPLETIONS_MODEL,
        messages=[
            {"role": "system", "content": f"You are a helpful assistant. Answer based on: {context}"},
            {"role": "user", "content": question}
        ],
        max_completion_tokens=1000
    )
    return completion.choices[0].message.content


scores = []
for example in eval_dataset:
    answer = generate_response(example["question"])

    scoring_prompt = (
        "Rate the answer's relevance to the question on a scale of 1 to 5.\n"
        "Respond with ONLY a single digit (1, 2, 3, 4, or 5). No words, no punctuation, no explanation.\n\n"
        f"Question: {example['question']}\n"
        f"Answer: {answer}\n"
        f"Ground truth: {example['ground_truth']}\n\n"
        f"Score (single digit only):"
    )

    completion = foundry_client.chat.completions.create(
        model=EVAL_MODEL,
        messages=[{"role": "user", "content": scoring_prompt}],
        max_completion_tokens=10
    )
    score_text = completion.choices[0].message.content.strip()

    print(f"Q: {example['question'][:40]}...")
    print(f"  Evaluating Answer: {answer[:60]}...")

    m = re.search(r"[1-5]", score_text)
    if m:
        score = int(m.group(0))
        scores.append(score)
        print(f"  Score: {score}/5")
    else:
        print(f"  Score text could not be parsed: {score_text}")
        scores.append(0)

if scores:
    avg = sum(scores) / len(scores)
    print(f"\nAverage relevance score: {avg:.2f}/5")
    print("Recommendation: score >4 = good, 3-4 = needs improvement, <3 = redesign RAG pipeline")